In [33]:
# !pip install snowflake-connector-python

In [32]:
# !pip install pyarrow

In [31]:
# !pip install "snowflake-connector-python[pandas]"

In [1]:
import os
import snowflake.connector
import pandas as pd
from snowflake.connector.pandas_tools import pd_writer,write_pandas

In [ ]:
def create_connection():
    conn = snowflake.connector.connect(
        user="hassanbutt",
        password="Hassan@4356",
        account="kggbypm-ir38077",
        warehouse="StreamLit_Data_House",
        database="Completion_Report",
        schema="SNOWFLAKE_SCHEMA"
    )
    return conn

In [36]:

# Function to create a connection to Snowflake
def create_snowflake_connection(schema):
    # Replace with your Snowflake credentials
    conn = snowflake.connector.connect(
        user="hassanbutt",  # Snowflake username
        password="Hassan@4356",  # Snowflake password
        account="kggbypm-ir38077",  # Snowflake account (e.g., abc12345.us-east-1)
        warehouse="StreamLit_Data_House",  # Snowflake warehouse name
        database="Completion_Report",  # Your Snowflake database name
        role="ACCOUNTADMIN",  # Your Snowflake database name
        schema=schema  # Schema for the current data load
    )
    return conn



In [33]:
def create_snowflake_connection():
    # Replace with your Snowflake credentials
    conn = snowflake.connector.connect(
        user="hassanbutt",  # Snowflake username
        password="Hassan@4356",  # Snowflake password
        account="kggbypm-ir38077",  # Snowflake account (e.g., abc12345.us-east-1)
        warehouse="StreamLit_Data_House",  # Snowflake warehouse name
        database="Completion_Report",  # Your Snowflake database name
        schema="public" , # Default schema (you can change it)
        role="ACCOUNTADMIN"  # Your Snowflake database name
    )
    return conn

In [34]:
conn=create_snowflake_connection()
cur=conn.cursor()

In [35]:
cur

In [6]:
table_name='Customers'

In [7]:
# Define the data
data = {
    "Customer ID": ["C001", "C002", "C003", "C004", "C005"],
    "Customer Name": ["Alice", "Bob", "Charlie", "Diana", "Ethan"]
}

# Create the DataFrame
df = pd.DataFrame(data)
df

,Customer ID,Customer Name
0,C001,Alice
1,C002,Bob
2,C003,Charlie
3,C004,Diana
4,C005,Ethan


In [8]:
cur.execute(f'CREATE TABLE {table_name} ("Customer ID" Varchar(30), "Customer Name" String)')

In [10]:
write_pandas(conn,df,table_name=table_name.upper())

(True,
 1,
 5,
 [('kbzrzcvalw/file0.txt', 'LOADED', 5, 5, 1, 0, None, None, None, None)])

In [11]:
sql_query=f'SELECT * FROM {table_name}'
cur.execute(sql_query)
data_fetch=cur.fetchall()

In [13]:
column_names = [desc[0] for desc in cur.description]

In [14]:
df = pd.DataFrame(data_fetch, columns=column_names)
df

,Customer ID,Customer Name
0,C001,Alice
1,C002,Bob
2,C003,Charlie
3,C004,Diana
4,C005,Ethan


In [15]:
column_names

['Customer ID', 'Customer Name']

In [41]:
def load_data_to_snowflake(file_path, sheet_name, schema, table_name):
    # Read data from Excel sheet
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    
    # Create a Snowflake connection for the specified schema
    conn = create_snowflake_connection(schema)
    
    try:
        # Set the schema for the current session
        conn.cursor().execute(f"USE SCHEMA {schema}")
        
        # Check if the table exists
        cursor = conn.cursor()
        cursor.execute(f"SHOW TABLES IN SCHEMA {schema}")
        tables = cursor.fetchall()
        table_names = [table[1] for table in tables]
        
        if table_name not in table_names:
            print(f"Table {table_name} does not exist in schema {schema}.")
        else:
            # Upload DataFrame to Snowflake table using write_pandas
            success, num_chunks, num_rows, error_msg = write_pandas(conn, df, table_name)
            if success:
                print(f"Data from sheet '{sheet_name}' loaded into {schema}.{table_name} successfully!")
            else:
                print(f"Error loading data to {schema}.{table_name}: {error_msg}")
    
    except Exception as e:
        print(f"Error loading data to {schema}.{table_name}: {str(e)}")
    finally:
        conn.close()


In [19]:
# # Function to create Snowflake table if not exists (or recreate if needed)
# def create_table_if_not_exists(conn, schema, table_name, df):
#     # Create table schema based on DataFrame columns (default type: STRING)
#     columns = ", ".join([f"{col} STRING" for col in df.columns])  # Assuming all columns are STRING, adjust types as needed
#     create_table_sql = f"""
#         CREATE TABLE IF NOT EXISTS {schema}.{table_name} (
#             {columns}
#         );
#     """
    
#     try:
#         cursor = conn.cursor()
#         cursor.execute(create_table_sql)
#         cursor.close()
#     except Exception as e:
#         print(f"Error creating table {schema}.{table_name}: {str(e)}")


In [20]:
# # Function to upload data from Excel to Snowflake
# def load_data_to_snowflake(file_path, sheet_name, schema, table_name):
#     # Read data from Excel sheet
#     df = pd.read_excel(file_path, sheet_name=sheet_name)
    
#     # Create a Snowflake connection
#     conn = create_snowflake_connection()

#     # Create table in Snowflake if not exists
#     create_table_if_not_exists(conn, schema, table_name, df)

#     # Optionally, clear existing data (TRUNCATE table before inserting new data)
#     truncate_sql = f"TRUNCATE TABLE {schema}.{table_name};"
#     try:
#         cursor = conn.cursor()
#         cursor.execute(truncate_sql)
#         cursor.close()
#     except Exception as e:
#         print(f"Error truncating table {schema}.{table_name}: {str(e)}")

#     # Upload DataFrame to Snowflake table
#     try:
#         write_pandas(conn, df, table_name)
#         print(f"Data from sheet '{sheet_name}' loaded into {schema}.{table_name} successfully!")
#     except Exception as e:
#         print(f"Error loading data to {schema}.{table_name}: {str(e)}")

In [16]:
# File path to your Excel file
file_path = 'reviewtool_20241224_VTA_RouteLevelComparison(Wkday & WkEnd)_Latest_01.xlsx'

# Dictionary of sheet names and corresponding schema and table names
sheet_info = {
    'WkDAY Route Comparison': ('wkday_comparison', 'route_comparison'),
    'WkDAY Route DIR Comparison': ('wkday_dir_comparison', 'route_dir_comparison'),
    'WkEND Route Comparison': ('wkend_comparison', 'route_comparison'),
    'WkEND Route DIR Comparison': ('wkend_dir_comparison', 'route_dir_comparison'),
    'WkEND Time Data': ('wkend_time_data', 'time_data'),
    'WkDAY Time Data': ('wkday_time_data', 'time_data')
#     'TOD': ('detail_data', 'tod_data')
}

# Iterate over all sheets and load data into corresponding Snowflake tables
# for sheet_name, (schema, table_name) in sheet_info.items():
#     load_data_to_snowflake(file_path, sheet_name, schema, table_name)

In [28]:
if  os.path.exists(file_path):
    for sheet_name, (schema, table_name) in sheet_info.items():
        print(sheet_name,(schema,table_name))

WkDAY Route Comparison ('wkday_comparison', 'route_comparison')
WkDAY Route DIR Comparison ('wkday_dir_comparison', 'route_dir_comparison')
WkEND Route Comparison ('wkend_comparison', 'route_comparison')
WkEND Route DIR Comparison ('wkend_dir_comparison', 'route_dir_comparison')
WkEND Time Data ('wkend_time_data', 'time_data')
WkDAY Time Data ('wkday_time_data', 'time_data')


In [43]:
import pandas as pd

# Example DataFrame
data = {
    'name': ['Alice', 'Bob', 'Charlie'],
    'age': [25, 30, 35],
    'salary': [50000, 60000, 70000]
}

# Create the DataFrame
df = pd.DataFrame(data)

# Get the column names with their data types
column_types = df.dtypes

# Print column names and data types
for column, dtype in column_types.items():
    print(f"Column: {column}, Data Type: {dtype}")


Column: name, Data Type: object
Column: age, Data Type: int64
Column: salary, Data Type: int64


In [22]:
# File path and dtype mapping
file_path = 'reviewtool_20241224_VTA_RouteLevelComparison(Wkday & WkEnd)_Latest_01.xlsx'
dtype_mapping = {
    'object': 'VARCHAR',
    'int64': 'INTEGER',
    'float64': 'FLOAT',
    'datetime64[ns]': 'TIMESTAMP',
    'bool': 'BOOLEAN'
}

# Dictionary of sheet names and corresponding table names
sheet_info = {
    'WkDAY Route Comparison': 'wkday_comparison', 
    'WkDAY Route DIR Comparison': 'wkday_dir_comparison', 
    'WkEND Route Comparison': 'wkend_comparison', 
    'WkEND Route DIR Comparison': 'wkend_dir_comparison', 
    'WkEND Time Data': 'wkend_time_data', 
    'WkDAY Time Data': 'wkday_time_data'
}


In [45]:
detail_df=pd.read_excel('details_vta_CA_od_excel.xlsx',sheet_name='TOD')
detail_df=detail_df[['OPPO_TIME[CODE]', 'TIME_ON[Code]', 'TIME_ON', 'TIME_PERIOD[Code]',
                              'TIME_PERIOD', 'START_TIME']]

In [31]:
table_name='TOD'
create_table_sql = f"CREATE TABLE IF NOT EXISTS {table_name} (\n"
for column, dtype in detail_df.dtypes.items():
    # Quote column names to handle special characters
    sanitized_column = f'"{column}"'
    snowflake_dtype = dtype_mapping.get(str(dtype), 'VARCHAR')  # Default to VARCHAR for unknown types
    create_table_sql += f"  {sanitized_column} {snowflake_dtype},\n"
create_table_sql = create_table_sql.rstrip(",\n") + "\n);"
create_table_sql

'CREATE TABLE IF NOT EXISTS TOD (\n  "OPPO_TIME[CODE]" VARCHAR,\n  "TIME_ON[Code]" VARCHAR,\n  "TIME_ON" VARCHAR,\n  "TIME_PERIOD[Code]" INTEGER,\n  "TIME_PERIOD" VARCHAR,\n  "START_TIME" VARCHAR\n);'

In [37]:
table_name='TOD'
create_table_sql = f"CREATE TABLE IF NOT EXISTS {table_name} (\n"
for column, dtype in detail_df.dtypes.items():
    # Quote column names to handle special characters
    sanitized_column = f'"{column}"'
    snowflake_dtype = dtype_mapping.get(str(dtype), 'VARCHAR')  # Default to VARCHAR for unknown types
    create_table_sql += f"  {sanitized_column} {snowflake_dtype},\n"
create_table_sql = create_table_sql.rstrip(",\n") + "\n);"
create_table_sql
cur.execute(create_table_sql)

# Insert data into the Snowflake table
write_pandas(conn, detail_df, table_name=table_name.upper())

(True,
 1,
 19,
 [('idsaogttby/file0.txt', 'LOADED', 19, 19, 1, 0, None, None, None, None)])

In [26]:
if os.path.exists(file_path):
    for sheet_name, table_name in sheet_info.items():
        # Read the sheet into a DataFrame
        df = pd.read_excel(file_path, sheet_name=sheet_name)

        # Dynamically generate the CREATE TABLE statement
        create_table_sql = f"CREATE TABLE IF NOT EXISTS {table_name} (\n"
        for column, dtype in df.dtypes.items():
            # Quote column names to handle special characters
            sanitized_column = f'"{column}"'
            snowflake_dtype = dtype_mapping.get(str(dtype), 'VARCHAR')  # Default to VARCHAR for unknown types
            create_table_sql += f"  {sanitized_column} {snowflake_dtype},\n"
        create_table_sql = create_table_sql.rstrip(",\n") + "\n);"
        print(create_table_sql)

        # Execute the CREATE TABLE statement
        cur.execute(create_table_sql)

        # Insert data into the Snowflake table
        write_pandas(conn, df, table_name=table_name.upper())

        print(f"Table {table_name} created and data inserted successfully.")

# Close the Snowflake connection
cur.close()
conn.close()


CREATE TABLE IF NOT EXISTS wkday_comparison (
  "ROUTE_SURVEYEDCode" VARCHAR,
  "Route Level Goal" FLOAT,
  "(0) Goal" FLOAT,
  "(1) Goal" FLOAT,
  "(2) Goal" FLOAT,
  "(3) Goal" FLOAT,
  "(4) Goal" FLOAT,
  "(5) Goal" FLOAT,
  "(0) Collect" INTEGER,
  "(1) Collect" INTEGER,
  "(2) Collect" INTEGER,
  "(3) Collect" INTEGER,
  "(4) Collect" INTEGER,
  "(5) Collect" INTEGER,
  "# of Surveys" INTEGER,
  "(0) Remain" INTEGER,
  "(1) Remain" INTEGER,
  "(2) Remain" INTEGER,
  "(3) Remain" INTEGER,
  "(4) Remain" INTEGER,
  "(5) Remain" INTEGER,
  "Remaining" INTEGER
);
Table wkday_comparison created and data inserted successfully.
CREATE TABLE IF NOT EXISTS wkday_dir_comparison (
  "ROUTE_SURVEYEDCode" VARCHAR,
  "(0) Goal" FLOAT,
  "(1) Goal" FLOAT,
  "(2) Goal" FLOAT,
  "(3) Goal" FLOAT,
  "(4) Goal" FLOAT,
  "(5) Goal" FLOAT,
  "(0) Collect" INTEGER,
  "(1) Collect" INTEGER,
  "(2) Collect" INTEGER,
  "(3) Collect" INTEGER,
  "(4) Collect" INTEGER,
  "(5) Collect" INTEGER,
  "(0) Remain"

C:\Users\hassa\AppData\Local\Programs\Python\Python310\lib\site-packages\pandas\io\parquet.py:190: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)


Table wkend_time_data created and data inserted successfully.
CREATE TABLE IF NOT EXISTS wkday_time_data (
  "Original Text" VARCHAR,
  "0" INTEGER,
  "1" INTEGER,
  "2" INTEGER,
  "3" INTEGER,
  "4" INTEGER,
  "5" INTEGER,
  "Time Range" VARCHAR,
  "Display_Text" INTEGER
);


C:\Users\hassa\AppData\Local\Programs\Python\Python310\lib\site-packages\pandas\io\parquet.py:190: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)


Table wkday_time_data created and data inserted successfully.


In [18]:
# Mapping Pandas dtypes to Snowflake SQL data types
dtype_mapping = {
    'object': 'VARCHAR',
    'int64': 'INTEGER',
    'float64': 'FLOAT',
    'datetime64[ns]': 'TIMESTAMP',
    'bool': 'BOOLEAN'
}


In [20]:
# Example DataFrame
data = {
    'name': ['Alice', 'Bob', 'Charlie'],
    'age': [25, 30, 35],
    'salary': [50000.0, 60000.5, 70000.0],
    'hire_date': pd.to_datetime(['2022-01-01', '2023-01-01', '2024-01-01']),
    'active': [True, False, True]
}
df = pd.DataFrame(data)

# Snowflake-compatible SQL statement generation
table_name = "example_table"
schema_name = "example_schema"

# Start creating the CREATE TABLE statement
create_table_sql = f"CREATE OR REPLACE TABLE {table_name} (\n"

# Iterate over DataFrame columns and their dtypes
for column, dtype in df.dtypes.items():
    snowflake_dtype = dtype_mapping.get(str(dtype), 'VARCHAR')  # Default to VARCHAR for unknown types
    create_table_sql += f"  {column} {snowflake_dtype},\n"

# Remove the trailing comma and close the statement
create_table_sql = create_table_sql.rstrip(",\n") + "\n);"




In [21]:
create_table_sql

'CREATE OR REPLACE TABLE example_table (\n  name VARCHAR,\n  age INTEGER,\n  salary FLOAT,\n  hire_date TIMESTAMP,\n  active BOOLEAN\n);'

In [38]:
# Table-to-DataFrame mapping
table_to_df_mapping = {
    'wkday_comparison': 'wkday_df',
    'wkday_dir_comparison': 'wkday_dir_df',
    'wkend_comparison': 'wkend_df',
    'wkend_dir_comparison': 'wkend_dir_df',
    'wkend_time_data': 'wkend_time_df',
    'wkday_time_data': 'wkday_time_df',
    'TOD': 'detail_df'
}

# Initialize an empty dictionary to hold DataFrames
dataframes = {}

# Loop through each table, fetch its data, and store it in the corresponding DataFrame
for table_name, df_name in table_to_df_mapping.items():
    # Query to fetch data
    query = f"SELECT * FROM {table_name}"
    
    # Fetch data from Snowflake
    cur = conn.cursor()
    cur.execute(query)
    data = cur.fetchall()
    
    # Get column names from the cursor description
    columns = [desc[0] for desc in cur.description]
    
    # Convert data to DataFrame
    df = pd.DataFrame(data, columns=columns)
    dataframes[df_name] = df
    
    print(f"Data fetched and stored in DataFrame: {df_name}")
    cur.close()

# Example: Access DataFrames
wkday_df = dataframes['wkday_df']
wkday_dir_df = dataframes['wkday_dir_df']
wkend_df = dataframes['wkend_df']
wkend_dir_df = dataframes['wkend_dir_df']
wkend_time_df = dataframes['wkend_time_df']
wkday_time_df = dataframes['wkday_time_df']
detail_df = dataframes['detail_df']

# Close the Snowflake connection
conn.close()

# Display example DataFrame
print(wkday_df.head())

Data fetched and stored in DataFrame: wkday_df
Data fetched and stored in DataFrame: wkday_dir_df
Data fetched and stored in DataFrame: wkend_df
Data fetched and stored in DataFrame: wkend_dir_df
Data fetched and stored in DataFrame: wkend_time_df
Data fetched and stored in DataFrame: wkday_time_df
Data fetched and stored in DataFrame: detail_df
  ROUTE_SURVEYEDCode  Route Level Goal  (0) Goal  (1) Goal  (2) Goal  \
0           VTA_2_20            79.625     0.000     0.000    15.375   
1           VTA_2_21           148.875     0.000     0.000    41.375   
2           VTA_2_22           687.500    21.125    11.875   159.375   
3           VTA_2_23           397.750     9.250     0.000    81.875   
4           VTA_2_25           385.000     4.875     1.625    84.750   

   (3) Goal  (4) Goal  (5) Goal  (0) Collect  (1) Collect  ...  (4) Collect  \
0    31.125    29.625     3.500            1            0  ...           13   
1    56.250    42.875     8.375            1            0  ..

In [39]:
detail_df

,OPPO_TIME[CODE],TIME_ON[Code],TIME_ON,TIME_PERIOD[Code],TIME_PERIOD,START_TIME
0,AM1,AM1,Before 5:00 am,0,OWL,01:00:00
1,AM2,AM2,5:00 am - 6:00 am,1,EARLY AM,05:00:00
2,AM3,AM3,6:00 am - 7:00 am,2,AM,06:00:00
3,MID1,MID1,7:00 am - 8:00 am,2,AM,07:00:00
4,MID2,MID2,8:00 am - 9:00 am,2,AM,08:00:00
5,MID7,MID7,9:00 am - 10:00 am,2,AM,09:00:00
6,MID3,MID3,10:00 am - 11:00 am,3,MID,10:00:00
7,MID4,MID4,11:00 am - 12:00 pm,3,MID,11:00:00
8,MID5,MID5,12:00 pm - 1:00 pm,3,MID,12:00:00
9,MID6,MID6,1:00 pm - 2:00 pm,3,MID,13:00:00


In [41]:
def fetch_dataframes_from_snowflake():
    """
    Fetches data from Snowflake tables and returns them as a dictionary of DataFrames.

    Returns:
        dict: A dictionary where keys are DataFrame names and values are DataFrames.
    """
    # Snowflake connection details
    conn = create_snowflake_connection()
    cur = conn.cursor()

    # Table-to-DataFrame mapping
    table_to_df_mapping = {
        'wkday_comparison': 'wkday_df',
        'wkday_dir_comparison': 'wkday_dir_df',
        'wkend_comparison': 'wkend_df',
        'wkend_dir_comparison': 'wkend_dir_df',
        'wkend_time_data': 'wkend_time_df',
        'wkday_time_data': 'wkday_time_df',
        'TOD': 'detail_df'
    }

    # Initialize an empty dictionary to hold DataFrames
    dataframes = {}

    try:
        # Loop through each table, fetch its data, and store it in the corresponding DataFrame
        for table_name, df_name in table_to_df_mapping.items():
            # Query to fetch data
            query = f"SELECT * FROM {table_name}"
            
            # Execute query and fetch data
            cur.execute(query)
            data = cur.fetchall()
            
            # Get column names from the cursor description
            columns = [desc[0] for desc in cur.description]
            
            # Convert data to DataFrame
            df = pd.DataFrame(data, columns=columns)
            dataframes[df_name] = df
            
            print(f"Data fetched and stored in DataFrame: {df_name}")
    except Exception as e:
        print(f"Error fetching data: {e}")
    finally:
        # Close cursor and connection
        cur.close()
        conn.close()

    return dataframes

# Fetch dataframes from Snowflake
dataframes = fetch_dataframes_from_snowflake()

Data fetched and stored in DataFrame: wkday_df
Data fetched and stored in DataFrame: wkday_dir_df
Data fetched and stored in DataFrame: wkend_df
Data fetched and stored in DataFrame: wkend_dir_df
Data fetched and stored in DataFrame: wkend_time_df
Data fetched and stored in DataFrame: wkday_time_df
Data fetched and stored in DataFrame: detail_df


In [42]:
wkday_df = dataframes.get('wkday_df')
wkday_dir_df = dataframes.get('wkday_dir_df')
wkend_df = dataframes.get('wkend_df')
wkend_dir_df = dataframes.get('wkend_dir_df')
wkend_time_df = dataframes.get('wkend_time_df')
wkday_time_df = dataframes.get('wkday_time_df')
detail_df = dataframes.get('detail_df')

In [43]:
detail_df

,OPPO_TIME[CODE],TIME_ON[Code],TIME_ON,TIME_PERIOD[Code],TIME_PERIOD,START_TIME
0,AM1,AM1,Before 5:00 am,0,OWL,01:00:00
1,AM2,AM2,5:00 am - 6:00 am,1,EARLY AM,05:00:00
2,AM3,AM3,6:00 am - 7:00 am,2,AM,06:00:00
3,MID1,MID1,7:00 am - 8:00 am,2,AM,07:00:00
4,MID2,MID2,8:00 am - 9:00 am,2,AM,08:00:00
5,MID7,MID7,9:00 am - 10:00 am,2,AM,09:00:00
6,MID3,MID3,10:00 am - 11:00 am,3,MID,10:00:00
7,MID4,MID4,11:00 am - 12:00 pm,3,MID,11:00:00
8,MID5,MID5,12:00 pm - 1:00 pm,3,MID,12:00:00
9,MID6,MID6,1:00 pm - 2:00 pm,3,MID,13:00:00
